# 00 — Environment check
Execute em um runtime Colab com GPU L4. O notebook clona/atualiza o repositório, monta o Drive, instala o projeto, registra o ambiente e oferece um smoke test quantizado opcional.

In [ ]:
from pathlib import Path
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
REPO_URL = 'https://github.com/devlucascfarias/logos-3.git'
PROJECT_ROOT = Path(os.environ.get('FABLE_PROJECT_ROOT', '/content/logos-3' if IN_COLAB else Path.cwd())).resolve()
DRIVE_ROOT = Path('/content/drive/MyDrive/fable-qwen-distillation') if IN_COLAB else PROJECT_ROOT
if IN_COLAB:
    if (PROJECT_ROOT / '.git').exists():
        dirty = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
        if dirty:
            print('Repositório local possui artefatos/alterações; pull automático ignorado para preservá-los.')
        else:
            subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
    elif PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'O destino existe e não é um clone Git vazio: {PROJECT_ROOT}')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(PROJECT_ROOT)], check=True)
print('project:', PROJECT_ROOT)
print('persistent:', DRIVE_ROOT)
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('O clone não contém pyproject.toml; confirme REPO_URL e a branch main')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

In [ ]:
import json, platform
from datetime import datetime, timezone
import torch, transformers, accelerate, peft, datasets

if not torch.cuda.is_available():
    raise RuntimeError('Selecione Runtime > Change runtime type > GPU')
free, total = torch.cuda.mem_get_info()
environment = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'gpu': torch.cuda.get_device_name(0),
    'vram_total_bytes': total,
    'vram_free_bytes': free,
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'transformers': transformers.__version__,
    'accelerate': accelerate.__version__,
    'peft': peft.__version__,
    'datasets': datasets.__version__,
}
print(json.dumps(environment, indent=2))
for directory in ('checkpoints/stage1', 'checkpoints/stage2', 'adapters', 'processed_data', 'datasets/raw', 'datasets/interim', 'data_manifests', 'merged', 'logs', 'evaluations'):
    (DRIVE_ROOT / directory).mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    links = {
        'data/raw': DRIVE_ROOT / 'datasets/raw',
        'data/interim': DRIVE_ROOT / 'datasets/interim',
        'data/processed': DRIVE_ROOT / 'processed_data',
        'data/manifests': DRIVE_ROOT / 'data_manifests',
        'outputs/checkpoints': DRIVE_ROOT / 'checkpoints',
        'outputs/adapters': DRIVE_ROOT / 'adapters',
        'outputs/merged': DRIVE_ROOT / 'merged',
        'outputs/logs': DRIVE_ROOT / 'logs',
        'outputs/evaluations': DRIVE_ROOT / 'evaluations',
    }
    for relative, target in links.items():
        local = PROJECT_ROOT / relative
        if local.is_symlink():
            continue
        existing = list(local.iterdir()) if local.exists() else []
        if any(item.name != '.gitkeep' for item in existing):
            raise RuntimeError(f'Refusing to replace non-empty artifact directory: {local}')
        for item in existing:
            item.unlink()
        if local.exists():
            local.rmdir()
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(target, target_is_directory=True)
(DRIVE_ROOT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')

In [ ]:
# Altere para True uma vez para validar download, quantização e forward pass.
RUN_MODEL_SMOKE = False
if RUN_MODEL_SMOKE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    name = 'Qwen/Qwen3-8B-Base'
    tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(name, quantization_config=quant, device_map={'': 0}, attn_implementation='sdpa', trust_remote_code=True)
    inputs = tokenizer('Write a Python function that adds two integers.', return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model(**inputs)
    print('forward logits:', tuple(output.logits.shape))
    del output, inputs, model
    torch.cuda.empty_cache()